#Initialization Step

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col 

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Data Transformations

## Trimming leading and trailing spaces

In [0]:
#Checking for possible issues in the dataset:
#Trim leading and trailing spaces
#Normalization for marital status: M = 'Married' & S = 'Single' and gndr: M = 'Male' & F = 'Female' 
#Column names are not user friendly

df.display()

In [0]:
#Makes it specific in which column you want to change
#df = df.withColumn('cst_firstname', trim(col('cst_firstname')))
#df = df.withColumn('cst_lastname', trim(col('cst_lastname'))

#This trims all columns with string type
for field in df.schema.fields: 
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
                           

#Normalization

In [0]:
#Syntax equivalent in SQL is CASE WHEN function

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .otherwise("n/a")
    )
)

In [0]:
df.display()

#Remove records with missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

#Renaming Columns 

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Sanity Checks of dataframe

In [0]:

df.limit(10).display()

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")


#Sanity Checks of Silver Table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10
